# box-array-to-tensor-with-recipe — worked example 2: Box an output when the grad gate is False (no Recipe)

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `box-array-to-tensor-with-recipe`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

The boxing step is **always** performed — every wrapped op returns a `MiniTensor`. But the `Recipe` is conditional. When the gate evaluates to False (no rg input, or global grad tracking off), the output is a graph **leaf**: `out.recipe` stays `None`. The reverse pass uses `recipe is None` to know it has hit the boundary and must stop, which is exactly the behaviour we want under inference / no_grad.

## Worked solution

We box the output of `square` applied to an input that does NOT require grad.

1. **Unbox.** `raw_args = (x.array,)`.
2. **Run forward.** `out_raw = x.array ** 2`, a plain array.
3. **Gate.** The single input has `requires_grad=False`, so `requires_grad = False`. (If a global toggle were involved it would AND in here too, but it cannot flip a False to True.)
4. **Box unconditionally.** `out = MiniTensor(out_raw, requires_grad=False)`. We still hand back a `MiniTensor` — boxing is never skipped.
5. **Skip the Recipe.** Since the gate is False, the `if requires_grad:` branch is not taken, so `out.recipe` keeps its constructor default of `None`. This marks `out` as a leaf and saves the bookkeeping cost of building a Recipe that backprop would never use.

In [ ]:
from typing import Callable
from dataclasses import dataclass

class MiniTensor:
    def __init__(self, array, requires_grad=False):
        self.array = np.asarray(array)
        self.requires_grad = requires_grad
        self.recipe = None

@dataclass
class Recipe:
    func: Callable
    args: tuple
    kwargs: dict
    parents: dict

def square(a):
    return a ** 2

def box_square(x: MiniTensor) -> MiniTensor:
    raw_args = (x.array,)
    requires_grad = x.requires_grad
    out_raw = square(*raw_args)
    out = MiniTensor(out_raw, requires_grad=requires_grad)
    if requires_grad:
        out.recipe = Recipe(square, raw_args, {}, {0: x})
    return out

x = MiniTensor(np.array([2.0, 3.0, 4.0]), requires_grad=False)
out = box_square(x)
print("array        :", out.array)
print("requires_grad:", out.requires_grad)
print("recipe is None:", out.recipe is None)